# 🧹 Module 3 — Class 1: Clean the Telco Data

**Lesson:** [https://bepro-aiml.github.io/aiml-platform/#/module/3/class/1](https://bepro-aiml.github.io/aiml-platform/#/module/3/class/1)

---

## 📖 Today's Story

It is Monday morning. You work at a telecom company in Tashkent.

Your boss walks in and says:

> *"We are losing customers. Tell me who will leave next month. Use our customer data."*

You think: *Easy! I learned ML. I will train a model.*

You open the file. **It is a mess.**

- The money column is text. (You can't add text!)
- Some rows are empty.
- One customer paid $99,999. (Real? Typo?)
- Same word in 3 ways: `Yes`, `yes`, `YES`.

**The model can not work with this data.**

So today we clean. Tomorrow we train.

---

## 💡 Why is cleaning important?

**Bad data → bad model → wrong answer → angry boss.**

Real data scientists spend **80% of their time** cleaning data. Not training models. Cleaning.

If you can clean data well, you can do ML.

If you can not, your model will be wrong. Always.

---

## 🤖 What is a notebook?

A notebook has two kinds of cells:
- 📝 **Text cells** (like this one) — they explain things.
- 💻 **Code cells** — Python code. You can run them.

**To run a code cell:** click on it. Press **Shift + Enter**.

## 🌐 First time using Google Colab?

Colab is free. No install. It works in your browser.

Click **Runtime → Run all** OR run cells one by one with **Shift + Enter**.

## 📦 Read the comments!

**Comments** are lines that start with `#`. Python skips them. They are notes for YOU.

**Read every comment.** They tell you what each line does.

Let's go! 🚀

---
## Step 0: Get the Tools

### 💡 Why?

Python alone can not work with tables. We need helpers (libraries).
- **pandas** = tables
- **numpy** = numbers
- **matplotlib** + **seaborn** = charts

We need to bring them in **one time** at the start.

In [ ]:
# 'import' = bring in the tool.
# 'as' = give it a short name.
import pandas as pd               # tables (DataFrames)
import numpy as np                # numbers
import matplotlib.pyplot as plt   # charts
import seaborn as sns             # nicer charts

# Hide small messages we don't need.
import warnings
warnings.filterwarnings('ignore')

print("Tools ready!")

---
## Step 1: Get the Customer Data

### 💡 Why?

Without data, no model. We need to load the customer file from the company.

We will use **Telco Customer Churn** data — about 7,000 real customers.

In [ ]:
# The file is online. Pandas can read from a URL.
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

# pd.read_csv() reads the file. We save the table in 'df'.
df = pd.read_csv(url)

# How big is the table?
print(f"We have {df.shape[0]} customers and {df.shape[1]} columns.")

# Show the first 5 rows. ALWAYS do this after loading data!
df.head()

👀 **Look at the table above.**

Each row = one customer. Each column = one fact about the customer.

Some columns:
- `tenure` = how many months the customer is with us
- `MonthlyCharges` = how much they pay each month
- `TotalCharges` = how much they paid all together
- `Churn` = did they leave? (Yes/No) ← **This is what we want to predict!**

---
## Step 2: Look Before You Touch

### 💡 Why?

Imagine you got a new car. **First thing?** You look around. Check the lights. Check the wheels. Smell the oil.

Same with data. **First we LOOK. Then we change.**

Three commands tell us a lot:

In [ ]:
# .info() shows: column name, type, and how many values are not empty.
#
# Look at 'Dtype':
#   'object'  = text
#   'int64'   = whole number (like 5)
#   'float64' = number with a dot (like 19.95)
df.info()

🚨 **PROBLEM!** Look at `TotalCharges`. It says `object` (text).

But `TotalCharges` is **money**. It should be a number.

We will fix this in Step 3. First, let's check more.

In [ ]:
# .describe() shows numbers about the NUMBER columns.
#   mean = average
#   min  = smallest value
#   max  = biggest value
# Always check min and max. Bad data hides there!
df.describe()

In [ ]:
# How many empty values in each column?
# True = empty. .sum() counts the True values.
df.isnull().sum()

### 🤔 Stop and think

1. How many rows? How many columns?
2. Which column is `object` but should be a number?
3. The biggest tenure is 72. Months or years? *(Hint: customers are not 72 years old!)*
4. Are there empty values?

---
## Step 3: Fix the Money Column 💰

### 💡 Why?

Right now `TotalCharges` is text. **You can not add text in math!**

Try this in your head: `"hello" + "world"` = `"helloworld"` (text)

But: `5 + 3` = `8` (number)

If the model sees `"100.50"` (text), it can not do anything with it. It crashes.

**We must change text to numbers.**

But why is it text? Let's find out!

In [ ]:
# Try to change to number. errors='coerce' = bad values become NaN (empty).
# Then .isna() finds those bad values.
mask = pd.to_numeric(df['TotalCharges'], errors='coerce').isna()

print(f"Bad rows: {mask.sum()}")
print("\nLet's see them:")
df.loc[mask, ['customerID', 'TotalCharges', 'tenure', 'MonthlyCharges']]

💡 **AHA!** Look — the bad rows all have `tenure = 0`.

These are **brand new customers**. They just signed up. They have not paid anything yet!

The `TotalCharges` value is **empty space** (" "). That is why pandas read everything as text.

**Now we know what to do:**
1. Change the column to numbers (empty becomes NaN).
2. NEW customers paid 0. So fill NaN with 0.

In [ ]:
# Step 1: Make it a number column. Bad values become NaN.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Step 2: Fill NaN with 0 (because new customers paid 0).
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Check: now it's a number, no empty values.
print("Type now:", df['TotalCharges'].dtype)
print("Empty values:", df['TotalCharges'].isna().sum())

### 🧠 Important lesson

We filled with **0**, not the median. Why?

Because **0 is the correct answer here.** A new customer paid 0.

**Always think:** *what does the empty value MEAN?*

Sometimes empty = unknown. Sometimes empty = zero. Sometimes empty = a real signal.

👉 **Domain knowledge wins. Talk to people who understand the business.**

---
## Step 4: Practice Filling Empty Values

### 💡 Why practice this?

**Real data ALWAYS has empty values.**

Real life examples:
- A customer didn't fill in their age on the form.
- A sensor was broken for 1 hour.
- The website crashed and didn't save the value.

**The model crashes with NaN. We must fill them.**

Three ways to fill:

| Way | When to use |
| --- | --- |
| **Mean** (average) | Numbers, no outliers |
| **Median** (middle value) | Numbers, safer with outliers |
| **Mode** (most common) | Text columns |

Our data is clean now. So we **make fake empty values** to practice.

We add 200 NaN to `MonthlyCharges` and try each way.

In [ ]:
# Make a copy. .copy() = new table, not connected to df.
# (Without .copy(), changing df_demo also changes df. Common mistake!)
np.random.seed(42)  # same random rows every time
df_demo = df.copy()

# Pick 200 random row numbers.
nan_idx = np.random.choice(df_demo.index, size=200, replace=False)

# Set them to NaN (empty).
df_demo.loc[nan_idx, 'MonthlyCharges'] = np.nan

print(f"Empty values now: {df_demo['MonthlyCharges'].isna().sum()}")

In [ ]:
# Way 1: MEAN (average)
mean_val = df_demo['MonthlyCharges'].mean()
df_mean = df_demo.copy()
df_mean['MonthlyCharges'] = df_mean['MonthlyCharges'].fillna(mean_val)
print(f"Mean: {mean_val:.2f}")

# Way 2: MEDIAN (middle value)
median_val = df_demo['MonthlyCharges'].median()
df_median = df_demo.copy()
df_median['MonthlyCharges'] = df_median['MonthlyCharges'].fillna(median_val)
print(f"Median: {median_val:.2f}")

# When do we pick mean vs median?
# If there is one VERY big value (like a $99,999 customer),
# the mean goes UP. Median stays the same. Median is safer.

### 📝 YOUR TURN — Way 3: Mode

**Mode** = the most common value.

**Why also add a 'missing' column?**

Sometimes 'this value is empty' tells us something!
- A customer who didn't fill in their phone → maybe doesn't trust us → maybe will leave.
- The fact of empty IS information. Save it!

### 📚 Hint

```python
# Mode
mode_val = df_demo['MonthlyCharges'].mode()[0]
df_mode['MonthlyCharges'] = df_mode['MonthlyCharges'].fillna(mode_val)

# Missing indicator (BEFORE fillna!)
df_mode['MonthlyCharges_missing'] = df_mode['MonthlyCharges'].isna().astype(int)
```

**WARNING:** Add the 'missing' column **BEFORE** you use fillna! After fillna, all values are not empty. Too late!

In [ ]:
df_mode = df_demo.copy()

# YOUR CODE — make 'MonthlyCharges_missing' column FIRST


# YOUR CODE — fill MonthlyCharges with mode


# Test (remove # after you write code):
# print(f"Mode: {mode_val:.2f}")
# print(df_mode['MonthlyCharges_missing'].value_counts())

---
## Step 5: Same Word, Same Meaning

### 💡 Why?

Look at this. We see in our data:
- `'No'`
- `'No internet service'`
- `'No phone service'`

All three mean **the customer does NOT have this thing**.

But the model thinks they are **3 different categories!**

It's like if I said: "Tashkent", "tashkent", "TASHKENT". Same city! But the model would think 3 cities.

**One meaning = one word.** Always.

In [ ]:
# First, find columns with the long version.
# We check each text column.
for col in df.select_dtypes(include='object').columns:
    # Get all different values
    vals = df[col].unique()
    # Keep values like 'No internet service'
    weird = [v for v in vals if 'No ' in str(v) and v != 'No']
    if weird:
        print(f"{col}: {weird}")

In [ ]:
# Now change them to plain 'No'.
# .replace(old, new) changes every match.

internet_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in internet_cols:
    df[col] = df[col].replace('No internet service', 'No')

df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

# Check: only 'Yes' and 'No' now
print("OnlineSecurity values now:", df['OnlineSecurity'].unique())
print("MultipleLines values now :", df['MultipleLines'].unique())

✅ Now the model sees only **2 categories** for these columns. Much better!

---
## Step 6: Find the Weird Numbers (Outliers)

### 💡 Why?

**An outlier = a number very different from the others.**

Real example: most customers pay $20-$100 per month. But what if one row says **$99,999**?

Three things it could be:

1. **A typo!** Someone pressed 9 too many times. Fix or remove.
2. **A real VIP customer.** Keep, but maybe cap to a max.
3. **Fraud!** This is what you WANT to find. **Don't remove!**

🚨 **Always think first: which kind do I have?**

Today we use the **IQR** rule. It's a simple, popular rule.

### 📚 The IQR rule

```
Q1   = 25% of values are below this
Q3   = 75% of values are below this
IQR  = Q3 - Q1
low  = Q1 - 1.5 × IQR
high = Q3 + 1.5 × IQR
anything outside [low, high] = outlier
```

In [ ]:
# A function = code with a name. We can use it many times.
# 'def' = make a new function.
def detect_outliers_iqr(series, factor=1.5):
    Q1 = series.quantile(0.25)        # 25% point
    Q3 = series.quantile(0.75)        # 75% point
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR         # the low limit
    upper = Q3 + factor * IQR         # the high limit
    # True/False list. True if outside [lower, upper].
    return (series < lower) | (series > upper), lower, upper

# Use the function on 3 number columns.
for col in ['MonthlyCharges', 'TotalCharges', 'tenure']:
    outliers, low, high = detect_outliers_iqr(df[col])
    print(f"{col}: {outliers.sum()} outliers (limits: [{low:.2f}, {high:.2f}])")

In [ ]:
# Box plot = the BEST chart to see outliers!
#
# What you see:
#   - Box      = middle 50% of values
#   - Line in box = middle value (median)
#   - Whiskers = normal range
#   - Dots     = OUTLIERS!
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(['MonthlyCharges', 'TotalCharges', 'tenure']):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

### 🤔 Stop and think

Look at the `tenure` chart. The biggest value is 72 months (= 6 years).

Is this customer a **mistake**? Or just a **loyal customer**?

👉 **A 6-year customer is REAL!** We do NOT remove them. We KEEP them.

Their long-time pattern can teach the model what loyal customers look like!

**Lesson:** *Big number ≠ wrong number.* Always think.

### 📝 OPTIONAL — Z-score method

Another way to find outliers. Not needed for today.

```python
z = (x - mean) / std
outlier if |z| > 3
```

---
## Step 7: Save the Clean Data

### 💡 Why save?

Tomorrow, we will **train a model** on this clean data.

If we don't save, we have to clean everything again. Waste of time!

**Rule:** Never write over the first file. Always save with a NEW name.

Why? If we made a mistake while cleaning, we can start over from the original.

In [ ]:
# Last check — is everything OK?
print("Shape:", df.shape)
print("Empty values:", df.isnull().sum().sum())
print("\nColumn types:")
print(df.dtypes.value_counts())

In [ ]:
# Save to a NEW file. index=False = don't save row numbers.
df.to_csv('telco_churn_cleaned.csv', index=False)
print("✅ Saved: telco_churn_cleaned.csv")

# In Colab — download to your computer:
# from google.colab import files
# files.download('telco_churn_cleaned.csv')

---
## 🏁 We Did It!

### What did we do today?

Look at the journey:

| Step | What | Why |
| --- | --- | --- |
| 1. Get data | Loaded from URL | Without data, no model |
| 2. Look | `.info()`, `.describe()` | Always look before you change |
| 3. Fix money | Text → number | Math doesn't work on text |
| 4. Fill empty | mean / median / mode | Model crashes with NaN |
| 5. Same words | `'No internet service'` → `'No'` | One meaning = one word |
| 6. Find weird numbers | IQR rule + box plots | Some numbers are wrong, some are gold |
| 7. Save | New CSV file | Tomorrow we train! |

### 🎯 Where are we going?

**Today:** Clean the data. ✅

**Class 2:** Make the data ready for the model (encoding, scaling).

**Class 3:** Build new useful columns (feature engineering).

**Module 4:** Train the model. Predict who will leave!

Each class makes the next class possible. 🪜

### 🎓 Words to Remember

- **DataFrame** — a table
- **NaN** — empty value
- **dtype** — type of a column
- **Mean** — average
- **Median** — middle value
- **Mode** — most common value
- **Outlier** — a value very different from the others
- **Domain knowledge** — knowing the business, not just the math

### 📤 Submit

Save this notebook as `Module3_Class1_<YourName>.ipynb`.

Push to your group repo at `module-3/class_1/submissions/<YourName>/`.

🧹 *Great job! See you in Class 2!*